# CESM2-LENS2 x blocking

PV-tendency budget closure at **250 hPa**, **dh = +0**, one cell per *(stage x RWB variant)*.
Each cell is **2 rows x 3 panels**: composite on top, a single event below;
columns are **LHS**, **RHS**, **LHS - RHS**.

$$\frac{\partial q}{\partial t} = -\left(u\frac{\partial q}{\partial x}
+ v\frac{\partial q}{\partial y} + \omega\frac{\partial q}{\partial p}\right) + Q$$

## Three traps this notebook is built to avoid

1. **`Q` is not the same quantity.** ERA5's is an independent latent-heating source
   (`tendency.py:945`). CESM2's `Q` is `Q_resid` = `dqdt - horiz_plus_vert`, a **residual by
   construction** — on the RHS it makes `LHS - RHS` identically zero and the closure look perfect.
   CESM2 uses `Q_lhr`.
2. **Units differ by 1e6.** ERA5 stores SI PV/s, CESM2 stores PVU/s.
3. **`levels` differ** (ERA5 9, CESM2 4), so 250 hPa is at a different index; looked up by VALUE.

Plus: CESM2's `mean_adv` **must** be added, because the ERA5 product list contains its mean-on-mean
members and the LHS is the full tendency in both.

## Known gaps

- ERA5 RWB variants live in `outputs/{blocking,prp}/rwb_variant_tracksets_wavg.pkl`; not yet
  wired in, so the ERA5 notebooks show a single `ALL` column.
- The sign convention of ERA5's stored products is undocumented, so it is **detected empirically**
  and printed. Check that line.

---

# The two RHS are NOT the same operator

Both archives test the same physics, but they discretise it differently, and the differences are
not cosmetic. Every line reference below is to the file that actually produced the numbers.

## ERA5 — `pvtend/src/pvtend/tendency.py`

**Advective form**, stored as raw un-summed products (the npz has 40 horizontal + 15 vertical):

$$\text{RHS} = -\sum_{a\in\{\bar{\ },'\}}\sum_{b\in\{\bar{\ },'\}}
\left(u_a\frac{\partial q_b}{\partial x} + v_a\frac{\partial q_b}{\partial y}
+ \omega_a\frac{\partial q_b}{\partial p}\right) + Q$$

12 products (2 wind states x 2 PV states x 3 directions). The mean-on-mean member
`u_bar_pv_bar_dx` **is** in the sum.

| quantity | formula | line |
|---|---|---|
| $Q$ | $-g(f+\zeta)\,\partial\dot\theta/\partial p$ | `tendency.py:945` |
| $\zeta$ | $\partial v/\partial x - \partial u/\partial y + u\tan\varphi/a$ | `tendency.py:942` |
| $\dot\theta$ | $\omega\left(\partial_p\theta - \frac{\gamma_m}{\gamma_d}\frac{\theta}{\theta_E}\partial_p\theta_E\right)$, **only where $\omega<0$** | `tendency.py:929-934` |
| $\gamma_m/\gamma_d$ | $\dfrac{1+L_Vq_s/(R_dT)}{1+L_V^2q_s/(c_pR_vT^2)}$ | `tendency.py:926-927` |

Winds are the **reanalysis winds**. Time step is **hourly**.

## CESM2 — `08_triad_resonance/build_pvbudget_15deg.py`

**Flux form for the rotational horizontal terms**, advective form for the divergent and vertical:

$$\text{rotational: } -\nabla\!\cdot\!(q\mathbf V) = -\frac{1}{a\cos\varphi}
\left[\frac{\partial(uq)}{\partial\lambda} + \frac{\partial(v\cos\varphi\,q)}{\partial\varphi}\right]
\qquad\text{`:426-431`}$$

$$\text{divergent, vertical: } -\left(u\frac{\partial q}{\partial x}+v\frac{\partial q}{\partial y}\right),\;
-\omega\frac{\partial q}{\partial p} \qquad\text{`:433-437`, `:502`}$$

| group | contents | line |
|---|---|---|
| `lin_adv` | $-\nabla\!\cdot\!(q'_p\bar{\mathbf V}_{rot}) - \nabla\!\cdot\!(q'_e\bar{\mathbf V}_{rot})$ | `:496` |
| `baroclinic` | $-\nabla\!\cdot\!(\bar q\mathbf V'_b) - \nabla\!\cdot\!(\bar q\mathbf V'_e)$ | `:497` |
| `rot_nl` | 4 terms $pp,pe,ep,ee$ | `:498-499` |
| `div_adv` | $-\bar{\mathbf V}_{div}\!\cdot\!\nabla q' - \mathbf V'_{div}\!\cdot\!\nabla\bar q - \mathbf V'_{div}\!\cdot\!\nabla q'$ | `:500-501` |
| `vertical` | $-\bar\omega\partial_pq' - \omega'\partial_p\bar q - \omega'\partial_pq'$ | `:502` |
| `mean_adv` | $-\bar{\mathbf V}\!\cdot\!\nabla\bar q - \bar\omega\partial_p\bar q$ | `:503` |
| `Q_lhr` | **identical in form to ERA5's `Q`** | `:164-177`, `:513-518` |

Time step is **daily**: `dqdt` is a 2-day centred difference `(pv[k+1]-pv[k-1])/2dt` (`:520`).

### Why `baroclinic` has 2 terms and `rot_nl` has 4 — not a missing cross term

`fadv` is **linear in the wind**. `baroclinic` splits only the wind (b + e); the advected field
$\bar q$ is not split, so $f(\mathbf V_b+\mathbf V_e,\bar q)=f(\mathbf V_b,\bar q)+f(\mathbf V_e,\bar q)$
holds exactly. Cross terms arise only when **both** factors are split — which is `rot_nl`.
Verified on the composites: `baroc_block+baroc_eddy-baroclinic` = 1.2e-07,
4 `rot_nl` terms minus `rot_nl` = 6.0e-08, `linadv_p+linadv_e-lin_adv` = 2.4e-07.

## THE DIFFERENCE THAT MATTERS MOST

**CESM2's $\mathbf V'_{rot}$ is not an observed wind.** It is the PPVI balanced reconstruction from
the **upper piece only** (`:300-325`) — measured against the archive Helmholtz field it
over-produces amplitude by ~10% (16.9 vs 15.3 m/s, corr 0.974) and omits the lower and surface
pieces (2.7 and 2.9 m/s) plus the unbalanced rotational flow. ERA5 uses the reanalysis wind
directly. So `baroclinic` and `rot_nl` are advection by a **balanced, partial** wind in CESM2 and by
the **actual** wind in ERA5.

## Measured: flux vs advective

CESM2 stores both (`horiz_plus_vert` and `horiz_plus_vert_adv`). Block, ALL, core box:

| stage | corr flux | corr adv | RMS resid flux | RMS resid adv |
|---|---|---|---|---|
| onset | 0.913 | **0.916** | 0.151 | **0.146** |
| peak | 0.692 | **0.736** | 0.110 | **0.109** |
| decay | 0.882 | **0.891** | 0.090 | **0.089** |

**The advective form closes marginally better at all three stages.** The flux form was adopted to
force the self-advection projection $\beta(\text{rot\_nl\_pp})\to0$. Measured:

| stage | $\beta$ CORE flux | CORE adv | FULL flux | FULL adv |
|---|---|---|---|---|
| onset | 0.1190 | 0.1241 | **0.0158** | 0.0145 |
| peak | 0.0465 | 0.0453 | **0.0205** | 0.0192 |
| decay | 0.0740 | 0.0780 | **0.0231** | 0.0260 |

The enstrophy argument $\int q\nabla\!\cdot\!(q\mathbf V) = -\tfrac12\int q^2\nabla\!\cdot\!\mathbf V$
requires **vanishing boundary flux**. Over the core box it does not vanish and $\beta\approx0.12$;
over the full patch $\beta$ drops by an order of magnitude to $\approx0.016$. So what drives
$\beta\to0$ is **using the full domain**, not the flux form — flux and advective are
indistinguishable at 0.0158 vs 0.0145.

---


In [ ]:
import sys, warnings
sys.path.insert(0, ".")
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from _closure import *

DATASET = "cesm2_blocking"
for st in STAGES:
    print(f"  {st:6s}  N={len(list_events(DATASET, st)):6,}  RWB={rwb_variants(DATASET, st)}")


## onset

In [ ]:
fig = cell(DATASET, "onset", rwb="ALL", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "onset", rwb="AWB", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "onset", rwb="CWB", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "onset", rwb="NEUTRAL", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "onset", rwb="Omega", event_index=0, limit=400)
plt.show()


## peak

In [ ]:
fig = cell(DATASET, "peak", rwb="ALL", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "peak", rwb="AWB", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "peak", rwb="CWB", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "peak", rwb="NEUTRAL", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "peak", rwb="Omega", event_index=0, limit=400)
plt.show()


## decay

In [ ]:
fig = cell(DATASET, "decay", rwb="ALL", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "decay", rwb="AWB", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "decay", rwb="CWB", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "decay", rwb="NEUTRAL", event_index=0, limit=400)
plt.show()


In [ ]:
fig = cell(DATASET, "decay", rwb="Omega", event_index=0, limit=400)
plt.show()
